# 3.5 — Named Entity Recognition (NER)

This notebook introduces Named Entity Recognition (NER) with spaCy: identifying people, organizations, locations, dates, monetary values, and other entities in text.

### Learning objectives
By the end of this notebook, you will be able to:
- load a spaCy language model and process text;
- extract named entities and interpret their labels;
- inspect entity positions in tokens and characters;
- create a custom entity;
- distinguish named entities from noun chunks;
- recognize the limitations of applying an English model to Arabic text.

## 1. Set up spaCy and the English model

In [ ]:
# If needed, run these commands in Jupyter or Colab.
# %pip install -U spacy
# !python -m spacy download en_core_web_sm

In [ ]:
import spacy

nlp = spacy....('en_core_web_sm')
print('spaCy model loaded successfully.')

spaCy model loaded successfully.


The `en_core_web_sm` model is trained for English. It can identify entities in English text, but it should not be treated as an Arabic NER model.

## 2. First NER analysis

A **named entity** is an expression referring to something such as a person, organization, location, date, or monetary value. spaCy stores detected entities in `doc.ents`.

In [ ]:
text = 'Apple plans to build a factory in Hong Kong for $6 million.'
doc = ...

for token in doc:
    print(token.text, end=' | ')

Apple | plans | to | build | a | factory | in | Hong | Kong | for | $ | 6 | million | . | 

In [ ]:
for ... in doc....:
    print(...text, '—', ...label_, '—', spacy....(....label_))

Apple — ORG — Companies, agencies, institutions, etc.
Hong Kong — GPE — Countries, cities, states
$6 million — MONEY — Monetary values, including unit


## 3. Inspect entity properties

In [ ]:
# Parcourir toutes les entités nommées détectées dans le document
for ent in doc.ents:
    # Afficher le texte correspondant à l'entité détectée
    print(f'Text: {ent.text}')
    # Afficher l'identifiant numérique du label de l'entité
    print(f'Label ID: {ent.label}')
    # Afficher le nom du label (PERSON, ORG, GPE, DATE, etc.)
    print(f'Label: {ent.label_}')
    # Afficher la signification du label à l'aide de spaCy
    print(f'Explanation: {spacy.explain(ent.label_)}')
    # Afficher la position de l'entité dans le document
    # start : indice du premier token
    # end   : indice du token suivant le dernier token de l'entité
    print(f'Token span: [{ent.start}, {ent.end})')
    # Afficher la position de l'entité en nombre de caractères
    # start_char : indice du premier caractère
    # end_char   : indice du caractère suivant le dernier caractère
    print(f'Character span: [{ent.start_char}, {ent.end_char})')
    print('-' * 40)

Text: Apple
Label ID: 383
Label: ORG
Explanation: Companies, agencies, institutions, etc.
Token span: [0, 1)
Character span: [0, 5)
----------------------------------------
Text: Hong Kong
Label ID: 384
Label: GPE
Explanation: Countries, cities, states
Token span: [7, 9)
Character span: [34, 43)
----------------------------------------
Text: $6 million
Label ID: 394
Label: MONEY
Explanation: Monetary values, including unit
Token span: [10, 13)
Character span: [48, 58)
----------------------------------------


**Pourquoi ce code est-il utile en NLP ?
Il permet de réaliser plusieurs opérations :
- Extraire automatiquement les noms de personnes et d'organisations.
- Identifier les pays, villes et localisations (GPE).
- Extraire les dates et les valeurs monétaires.
- Retrouver précisément une entité dans le texte original.
- Construire des systèmes d'extraction d'informations (Information Extraction).
À retenir  :
doc.ents contient les entités nommées détectées par spaCy. Chaque ent est un objet Span qui fournit le texte de l'entité, sa catégorie grammaticale sémantique et sa position exacte dans le document.

## 4. Create a reusable helper function

The function prints detected entities, or a message if none are found.

In [ ]:
def show_ents(doc):
    #Vérifier si spaCy a détecté au moins une entité nommée dans le document.
    if .....:
        for ... in doc....:
            print(f'{ent.text} — {ent.label_} — {spacy.explain(ent.label_)}')
    else:
        print('No named entities found.')

## 5. Test several examples

In [ ]:
examples = [
    'Hi, how are you?',
    'May I go to Washington, DC next May to see the Washington Monument?',
    'Can I borrow 500 dollars to buy Microsoft stock?',
    'CPRO plans to build a U.K. factory for $6 million.'
]

for text in examples:
    print(f'\nTEXT: {text}')
    show_ents(....)


TEXT: Hi, how are you?
No named entities found.

TEXT: May I go to Washington, DC next May to see the Washington Monument?
Washington, DC — GPE — Countries, cities, states
next May — DATE — Absolute or relative dates or periods
the Washington Monument — ORG — Companies, agencies, institutions, etc.

TEXT: Can I borrow 500 dollars to buy Microsoft stock?
500 dollars — MONEY — Monetary values, including unit
Microsoft — ORG — Companies, agencies, institutions, etc.

TEXT: CPRO plans to build a U.K. factory for $6 million.
U.K. — GPE — Countries, cities, states
$6 million — MONEY — Monetary values, including unit


## 6. Add a custom entity

A model may not recognize rare or domain-specific names. You can add an entity manually with `Span`, then reassign `doc.ents`. Span indices use a half-open interval: the end index is excluded.

In [ ]:
from spacy.tokens import Span

text = 'CPRO plans to build a U.K. factory for $6 million.'
doc = ...

print('Before adding the custom entity:')
show_ents(doc)

# CPRO is the first token in this example.
org_label = doc.vocab.strings['ORG']
custom_ent = Span(doc, 0, 1, label=org_label)

# Remove any model-predicted entity that overlaps the first token,
# then add the custom entity.
kept_ents = [ent for ent in doc.ents if ent.end <= 0 or ent.start >= 1]
doc.ents = tuple(kept_ents) + (custom_ent,)

print('\nAfter adding the custom entity:')
show_ents(doc)

## 7. Noun chunks

`noun_chunks` are syntactic noun phrases, not named entities. For example, “autonomous cars” is a noun chunk even though it is not a proper name.

In [ ]:
text = 'Autonomous cars shift insurance liability toward manufacturers.'
doc = nlp(text)

for chunk in doc.noun_chunks:
    print(f'Chunk: {chunk.text}')
    print(f'Root: {chunk.root.text}')
    print(f'Root dependency: {chunk.root.dep_} — {spacy.explain(chunk.root.dep_)}')
    print(f'Token span: [{chunk.start}, {chunk.end})')
    print(f'Character span: [{chunk.start_char}, {chunk.end_char})')
    print('-' * 40)

## 8. Analyze a longer text

In [ ]:
text = '''
Inflation is a general rise in the price level in an economy over time.
When prices rise, each unit of currency buys fewer goods and services.
Economists study the causes and effects of inflation and deflation.
'''
doc_long = nlp(text)

print('Named entities:')
show_ents(doc_long)

print('\nNoun chunks:')
for chunk in doc_long.noun_chunks:
    print(f'{chunk.text} -> root: {chunk.root.text}')

## 9. Arabic text: limitations of the English model

The `en_core_web_sm` model is not suitable for Arabic linguistic analysis. To tokenize Arabic text only, you can use `spacy.blank('ar')`. This does not, by itself, provide reliable named entity recognition or noun-chunk parsing.

In [ ]:
arabic_text = 'شركة مرسيدس في ألمانيا استحوذت على شركة كرايسلر مقابل خمسة ملايين دولار.'

# Tokenization only; this blank pipeline has no trained NER component.
nlp_ar = spacy.blank('ar')
doc_ar = nlp_ar(arabic_text)
print('Arabic tokens:')
print(' | '.join(token.text for token in doc_ar))

print('\nEntities detected by this blank Arabic pipeline:', list(doc_ar.ents))

### Practice

1. Run the model on a sentence containing a person, an organization, and a location.
2. Print each entity's text, label, and label explanation.
3. Compare `ent.start`, `ent.end`, `ent.start_char`, and `ent.end_char`.
4. Manually add an `ORG` entity that the model missed.
5. Explain the difference between `doc.ents` and `doc.noun_chunks`.
6. Why should you not interpret the English model's output on Arabic text as reliable NER?

In [ ]:
# Exercise 1: replace this sentence with your own example.
text = 'Mary works at Google in London.'
doc = nlp(text)
show_ents(doc)

# Exercise 2: print the entity spans.
for ent in doc.ents:
    print(ent.text, ent.start, ent.end, ent.start_char, ent.end_char)

# Exercise 3: inspect noun chunks.
for chunk in doc.noun_chunks:
    print(chunk.text)

### Key takeaways
- `doc.ents` contains the entities recognized by the model.
- `ent.text` is the entity text; `ent.label_` is its readable label.
- `spacy.explain(ent.label_)` explains the label.
- `Span(doc, start, end, label=...)` can create a custom entity.
- `doc.noun_chunks` contains noun phrases, not named entities.
- Analysis quality depends on the model and the language of the text.

### Useful references
- spaCy documentation: https://spacy.io/usage/linguistic-features#named-entities
- `Span` API: https://spacy.io/api/span